## Structured Output in LangChain
Models can be instructed to return responses in a specific format that matches a predefined schema. This makes the output easier to validate, parse, and use in subsequent processing.
##### Key Points
- Schema: Defines the expected structure, fields, and data types of the output.
- Validation: Ensures the returned data follows the required format.
- Parsing: Converts the model's response into a structured representation, such as a Python object.
- Use cases: API responses, data extraction, information processing, and automation.
- LangChain supports multiple approaches to structured output, depending on the model and integration.



## Pydantic
Pydantic is a Python library used to define data schemas and validate data using Python type hints.
##### Key Features
- Field validation: Checks whether values match the expected types and constraints.
- Descriptions: Adds explanations to fields to clarify their purpose.
- Nested structures: Supports complex data models containing other models.
- Type safety: Helps ensure data follows the defined schema.

In [41]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:openai/gpt-oss-120b")

In [42]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The rating of the movie out of 10")
    

In [43]:
model_with_structure = model.with_structured_output(Movie)

In [44]:
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023D043C88D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000023D043C9260>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The ti

In [45]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception (2010) – Quick Reference**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Date** | July\u202f16, 2010 (USA) |\n| **Running Time** | 148 minutes |\n| **Genre** | Science‑fiction, Action, Thriller, Heist |\n| **Director / Writer** | Christopher Nolan |\n| **Producers** | Emma Thomas, Christopher Nolan, Michael Caine (executive), Charles Roven (executive) |\n| **Production Companies** | Warner Bros. Pictures, Legendary Pictures, Syncopy Inc. |\n| **Budget** | ≈\u202f$160\u202fmillion (USD) |\n| **Box‑Office Gross** | ≈\u202f$836\u202fmillion worldwide |\n| **MPAA Rating** | PG‑13 (Violence, some language, brief strong drug content) |\n| **Primary Cast** | • **Leonardo DiCaprio** – Dom Cobb (extractor) <br>• **Ken Watanabe** – Saito (client) <br>• **Joseph Gordon‑Levitt** – Arthur (point man) <br>• **Marion Cotillard** – Mal Cobb (Dom’s late wife) <br>• **Elliot Page** – Ariadne (architect) <br>• **Tom Hardy** – Eames (for

In [46]:
model_with_structure.invoke("Provide details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

#### Raw Message  output alongside parsed output structure

In [47]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(...,description="The title of the movie")
    year:int=Field(...,description="The year the movie was released")
    director:str=Field(...,description="The director of the movie")
    rating:float=Field(...,description="The rating of the movie out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the movie Inception. We have a function to get movie details: title, director, rating, year. We should call it.', 'tool_calls': [{'id': 'fc_40f046b0-afec-4d44-807a-be3c1e9ac163', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 84, 'prompt_tokens': 165, 'total_tokens': 249, 'completion_time': 0.184181503, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.006936687, 'prompt_tokens_details': None, 'queue_time': 0.393334607, 'total_time': 0.19111819}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a183c4-acb2-7387-8ce9-ef23c6a7f004-0', tool_calls=[{'name': 'Movie', 'args': {'

#### Nested Structure

In [48]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails(BaseModel):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="Budget in USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year='2010', cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Dileep Rao', role='Yusuf'), Actor(name='Tom Berenger', role='Bob')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

#### Using with Agent

In [59]:
# groq:openai/gpt-oss-120b
import os
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name:str
    email:str
    phone:str

agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


## TypedDict
TypedDict provides a way to define the expected structure of a dictionary using Python's built-in typing module. It is ideal when you need type hints but don't need runtime validation.
#### Key Points
- Defines the expected keys and value types of a dictionary.
- Helps IDEs and static type checkers detect type-related mistakes.
- Values are accessed using dictionary keys.
- Does not enforce type validation at runtime by default.
- Useful for defining structured data without creating a full class.

In [50]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title:Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The rating of the movie out of 10"]

model_with_typedDict = model.with_structured_output(MovieDict)

response=model_with_typedDict.invoke("Provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

#### Nested Structure

In [51]:
class Actor(TypedDict):
    name:str
    role:str

class MovieDetails(TypedDict):
    title:str
    year:str
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None,description="Budget in USD")

model_with_typedDict = model.with_structured_output(MovieDetails)

response = model_with_typedDict.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Gwyneth Paltrow', 'role': 'Pepper Potts'},
  {'name': 'Paul Bettany', 'role': 'J.A.R.V.I.S. (voice)'},
  {'name': 'Robert Redford', 'role': 'Alexander Pierce'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'title': 'The Avengers',
 'year': '2012'}

#### Using with Agent

In [52]:
# groq:openai/gpt-oss-120b
import os
from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name:str
    email:str
    phone:str
    
agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [53]:
# model_with_typedDict.profile   # error if the model uses structured output

In [54]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

## DataClasses
A dataclass is a Python class designed primarily to store data. It is created using the @dataclass decorator from the dataclasses module.
#### Key Points
- Automatically generates methods such as __init__() and __repr__().
- Uses type annotations to define data fields.
- Reduces boilerplate code when creating data-holding classes.
- Does not automatically enforce type validation at runtime.

#### Using with agent

In [57]:
# groq:openai/gpt-oss-120b
import os
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name:str
    email:str
    phone:str
    
agent = create_agent(
    model = "groq:openai/gpt-oss-120b",
    response_format=ContactInfo
)

result = agent.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content":"extract contact info from: John Doe, john@example.com, (555) 123-4567"
            }
        ]
    }
)

print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [58]:
result

{'messages': [HumanMessage(content='extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='bddf736c-966f-4cc2-9e71-335514f19073'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'reasoning_content': 'The user wants extraction of contact info. The response format is a JSON object matching the ContactInfo schema: properties name, email, phone, all required. Need to output compact JSON with those fields.\n\nWe need to ensure it\'s valid JSON, compact (no whitespace). Provide:\n\n{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}\n\nCheck required fields present. Yes.\n\nReturn only JSON.'}, response_metadata={'token_usage': {'completion_tokens': 126, 'prompt_tokens': 212, 'total_tokens': 338, 'completion_time': 0.264136638, 'completion_tokens_details': {'reasoning_tokens': 90}, 'prompt_time': 0.011491614, 'prompt_tokens_details': None, 'q